In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/zalando-research/fashionmnist/t10k-labels-idx1-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/t10k-images-idx3-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_test.csv
/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv
/kaggle/input/datasets/zalando-research/fashionmnist/train-labels-idx1-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/train-images-idx3-ubyte


In [17]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms
import torch.optim as optim
import torchvision.models as models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
torch.manual_seed(42)

In [3]:
df = pd.read_csv("/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv")


In [4]:
x = df.iloc[:, 1 : ]. values
y = df.iloc[:, 0].values

In [5]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [9]:
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [45]:
class CustomDataset(Dataset):
    def __init__(self, features, labels, transform):
        self.features = features
        self.labels = labels
        self.transform = transform 

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        image = self.features[index].reshape(28,28)
        image = image.astype(np.uint8)
        image = np.stack([image]*3, axis = -1)
        image = Image.fromarray(image)
        image = self.transform(image)
        return image, torch.tensor(self.labels[index], dtype = torch.long)

In [46]:
train_dataset = CustomDataset(X_train, y_train, transform = custom_transform)
len(train_dataset)

48000

In [47]:
test_dataset = CustomDataset(X_test, y_test, transform = custom_transform)

In [48]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False, pin_memory = True)

In [49]:
vgg16 = models.vgg16(pretrained = True)

In [50]:
for param in vgg16.features.parameters():
    param.requires_grad = False

In [51]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10),
    nn.ReLU(),
)

In [52]:
vgg16 = vgg16.to(device)

In [53]:
EPOCHS = 10
LEARNING_RATE = 0.0001

In [54]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters(), lr = LEARNING_RATE)

In [55]:
len(train_loader)

1500

In [56]:
for epoch in range(EPOCHS):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = vgg16(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch:{epoch+1}, Loss:{avg_loss}')

Epoch:1, Loss:0.39570397601028284
Epoch:2, Loss:0.22094682429234186
Epoch:3, Loss:0.17098186229976514
Epoch:4, Loss:0.13542755187939232
Epoch:5, Loss:0.10707056702983876
Epoch:6, Loss:0.08287468292375949
Epoch:7, Loss:0.0707251745845812
Epoch:8, Loss:0.05653681306175228
Epoch:9, Loss:0.049094196482289895
Epoch:10, Loss:0.04266467124353706


In [57]:
torch.save(model.state_dict(), '/kaggle/working/my_trained_model.pth')


In [58]:
model.load_state_dict(torch.load('/kaggle/working/my_trained_model.pth'))
model.eval()
total = 0
correct = 0
y_pred = []
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = vgg16(batch_features)
        _, predicted = torch.max(outputs, 1)
        y_pred.append(predicted)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.9154166666666667
